## Minimum-Biased Weighted Average
### Experiment notebook

### Calcolo di Gamma omicron con il metodo della <i>Media pesata orientata ai minimi</i>

Dato un vettore di subgrades $s = (s_1, s_2, ..., s_n)$, l'equazione 
$$
{\huge
\Gamma\omicron = \frac {\displaystyle\sum_{i=1}^n s_i * w_i} {\displaystyle\sum_{i=1}^n w_i}
}
\hspace{4cm}(1)
$$
con
$$
{\huge
w_i = \frac{1}{s_i^p}
}
\hspace{4cm}(2)
$$
produce un valore Gamma omicron - sinonimo di "Overall Grade" - che riflette il netto scostamento della media dei valori verso i minimi.

Il valore ideale dell'esponente $p$ è stato identificato empiricamente in un range prossimo al valore 2.

### Introduzione del coefficiente di severità del difetto

Versione alternativa con coefficiente $\alpha_i$. Alpha-iesimo è specifico della tipologia di sottovoto e determina il peso del sottovoto in funzione della sua severità percepita. Ad esempio, un graffio superficiale viene percepito come più un difetto più severo di una scoloritura, etc.

$$
{\huge
w_i = \alpha_i\frac{1}{s_i^p}
}
\hspace{4cm}(3)
$$

Hint: la lettera greca ideale per severity è forse $\sigma$ (sigma)?

### Introduzione del centering

Un centering non preciso riduce il valore di $\Gamma\omicron$ di un intervallo di valori tra 0 e 1 quando $\Gamma\omicron$ è compreso tra 1 e 10. Si noti che l'impatto del centering sale al valore 1.5 quando $\Gamma\omicron = 10.5$ (10 Pristine), come effetto della promozione automatica della carta "perfetta" - tutti i subgrades a 10 e i due centering a 50-50 - a "pristine".

Indichiamo con la lettera greca $X$ (CHi) il centering e quindi:
$X\omicron$ (Chi omicron) è il centering overall, risultante da

$$
{\huge
X\omicron = tdb(\chi_{front}, \chi_{back})
}
\hspace{4cm}(4)
$$

$$
{\huge
\Gamma\omicron = X\omicron \frac {\displaystyle\sum_{i=1}^n s_i * w_i} {\displaystyle\sum_{i=1}^n w_i}
}
{\huge
\hspace{0.5cm}con\hspace{0.5cm}
w_i = \alpha_i\frac{1}{s_i^p}
}
{\huge
\hspace{0.5cm}dove\hspace{0.5cm}
0.9 \leq X\omicron \leq 1
\hspace{4cm}(5)
}
$$

In [59]:
"""Boilerplate code and utility functions"""

def separator(length=50):
    """return a string for separation of titles and text""" 
    row = "\n"
    for i in range(length):
        row += "-"
    return row

def title(text):
    """return a formatted title"""
    return "\n" + text + separator(50)

In [3]:
import sys
import numpy as np
from IPython.display import display, Math, Latex
from typing import Any, BinaryIO, Generic, Iterable, MutableSequence, Tuple, TypeVar, Union, overload
from typing_extensions import Literal

def weight(val, exponent):
    return 1 / (val**exponent)

def min_biased_weighted_avg_v1(s: MutableSequence):
    prod = 0.0
    wsum = 0.0
    exponent = 2 
    # calculate sum of si * wi
    for i, xi in enumerate(s):
        wi = weight(xi, exponent)
        prod += (xi * wi)
    # calculate the sum of all weights
    for index, wi in enumerate(s):
        wsum += weight(wi, exponent)
    avg_v1 = prod / wsum
    return avg_v1

subgrades = [2, 3, 9, 3, 4, 8]
avg = np.mean(subgrades)
avg_biased = min_biased_weighted_avg_v1(subgrades)
print("Esempio\n------------------------------------")
print(f"subgrades = {subgrades}")
print(f"media matematica = {avg:0.5f}")
print(f"media min-biased = {avg_biased:0.5f}")

Esempio
------------------------------------
subgrades = [2, 3, 9, 3, 4, 8]
media matematica = 4.83333
media min-biased = 2.93726


In [6]:
import sys

def weight(val, exponent):
    return 1 / (val ** exponent)

def min_biased_weighted_avg_v2(s, exponent=2):
    """super compact code, good speed, worst legibility"""
    weights = [weight(x, exponent) for x in s]
    wsum = sum(weights)
    prod = sum(x * w for x, w in zip(s, weights))
    return prod / wsum
    
subgrades = [2, 3, 9, 3, 4, 8]
avg = np.mean(subgrades)
avg_biased = min_biased_weighted_avg_v2(subgrades)
print("Esempio\n------------------------------------")
print(f"subgrades = {subgrades}")
print(f"media matematica = {avg:0.5f}")
print(f"media min-biased = {avg_biased:0.5f}")

Esempio
------------------------------------
subgrades = [2, 3, 9, 3, 4, 8]
media matematica = 4.83333
media min-biased = 2.93726


In [2]:
from ipywidgets import interact
import ipywidgets as widgets
from ipywidgets import interactive,interact, HBox, Layout, VBox

import math

def demo(s1, s2, s3, s4, s5, s6, exponent=2):
    s = [s1, s2, s3, s4, s5, s6,]
    weights = [1/(x**exponent) for x in s]
    G = sum(x*w for x, w in zip(s, weights)) / sum(weights)
    #rint(f"s = {s}")
    print(f"pesi = {[round(w, 1) for w in weights]}")
    print(f"G = {G:.5f}")
    quantization = min(10, math.ceil(2*G)/2)
    print(f"Gq = {quantization:0.1f}")

widget = interactive(
    demo,
    s1=widgets.FloatSlider(value=2, min=1, max=10, step=0.5),
    s2=widgets.FloatSlider(value=3, min=1, max=10, step=0.5),
    s3=widgets.FloatSlider(value=9, min=1, max=10, step=0.5),
    s4=widgets.FloatSlider(value=9, min=1, max=10, step=0.5),
    s5=widgets.FloatSlider(value=9, min=1, max=10, step=0.5),
    s6=widgets.FloatSlider(value=9, min=1, max=6, step=0.5),
    exponent=widgets.FloatSlider(value=2, min=1, max=10, step=0.2))

box1 = VBox(widget.children[0:3])
box2 = VBox(widget.children[3:6])
box3 = VBox(widget.children[6:7])
box = HBox([box1, box2, box3])
output = widget.children[-1]
display(VBox([box, output]))



$$
{\huge
\Gamma\omicron = X\omicron \frac {\displaystyle\sum_{i=1}^n s_i * w_i} {\displaystyle\sum_{i=1}^n w_i}
 , 
w_i = \frac{1}{s_i^p}
}
,
0.9 \leq X\omicron \leq 1
$$

In [2]:
from ipywidgets import interact
import ipywidgets as widgets
from ipywidgets import interactive, interact, HBox, Layout, VBox
import numpy as np
import matplotlib.pyplot as plt

def G(s, p):
    s = np.asarray(s, dtype=float)
    w = 1.0 / (s ** p)
    return np.sum(s * w) / np.sum(w)

def weights(s, p):
    s = np.asarray(s, dtype=float)
    w = 1.0 / (s ** p)
    return w / np.sum(w)  # normalize weights to 1

def quantize_half_up_to_10(x):
    return min(10.0, np.ceil(2.0 * x) / 2.0)

def quantize_half_down_to_10(x):
    return min(10.0, np.floor(2.0 * x) / 2.0)

def quantize_mik(x):
    frac = x % 1
    if frac < 0.25:
        return np.floor(x)
    if frac >= 0.75:
        return np.ceil(x)
    return np.floor(x) + 0.5 

def new_slider(val, minval, maxval, stepval, desc, update=False):
    w = widgets.FloatSlider(value=val, min=minval, max=maxval, step=stepval,
                            description=desc,
                            continuous_update = update,
                            style={'description_width': 'initial'},
                            layout=widgets.Layout(width='35%'))
    return w

def sandbox(s1, s2, s3, s4, s5, s6,xo, p=2.0):
    arr = [s1, s2, s3, s4, s5, s6]
    s = np.array(arr, dtype=float)

    # Curva G(p) su un range
    p_min, p_max = 0.0, 10.0
    ps = np.linspace(p_min, p_max, 400)
    Gs = np.array([G(s, pi) for pi in ps])

    g = G(s, p)
    g *= xo
    gq_ceil = quantize_half_up_to_10(g)
    gq_floor = quantize_half_down_to_10(g)
    gq_mik = quantize_mik(g)
    wn = weights(s, p)

    # ---- Plot scoring
    plot_scoring = plt.figure(figsize=(8, 3))
    plt.bar(range(1, len(s)+1), s)
    plt.yticks(range(0, 11), [f"{i}" for i in range(0, 11)]) 
    plt.xticks(range(1, len(s)+1), [f"s{i}" for i in range(1, len(s)+1)])
    plt.title("Scoring assoluto")
    plt.ylabel("scoring")
    plt.show()

    # ---- Plot pesi normalizzati (chi domina)
    plot_weights = plt.figure(figsize=(8, 3))
    plt.bar(range(1, len(s)+1), wn, color="orange")
    plt.xticks(range(1, len(s)+1), [f"s{i}" for i in range(1, len(s)+1)])
    plt.ylabel("peso normalizzato")
    plt.title("Pesi normalizzati a p corrente - chi Domina ")
    plt.show()

    # ---- Plot G(p)
    plt.figure(figsize=(8, 3))
    plt.plot(ps, Gs)
    plt.axhline(np.min(s), color='grey', linestyle=":") 
    plt.axhline(np.mean(s), color='grey', linestyle="-")
    plt.axhline(np.max(s), color='grey', linestyle=":")
    plt.axhline(y=gq_ceil, color='red', linestyle='-')
    plt.axhline(y=gq_floor, color='red', linestyle='-')
    plt.axhline(y=g, color="green", linestyle="-")
    plt.axhline(y=gq_mik, color='orange', linestyle=':')
    plt.scatter([p], [g])
    plt.xlabel("p")
    plt.ylabel("G(p)")
    #plt.yticks(np.arange(0, 11, 0.5))
    plt.title("Min-biased weighted average: G(p)")

    plt.show()
    
    print("\n")
    print(f"s = {list(np.round(s, 3))}")
    print(f"p = {p:.3f}")
    print(f"G = {g:.5f}")
    print(f"Gq round up= {gq_ceil:.1f}")
    print(f"Gq round down= {gq_floor:.1f}")
    print(f"Gq round down= {gq_floor:.1f}")
    print(f"Gq Mik= {gq_mik:.1f}")
    
sliders = widgets.interactive(
    sandbox,
    s1=new_slider(val=6, minval=1, maxval=10, stepval=0.5, desc="s1 Front Surface"),
    s2=new_slider(val=7, minval=1, maxval=10, stepval=0.5, desc="s1 Front Edges"),
    s3=new_slider(val=9, minval=1, maxval=10, stepval=0.5, desc="s1 Front Corners"),
    s4=new_slider(val=6, minval=1, maxval=10, stepval=0.5, desc="s1 Back Surface"),
    s5=new_slider(val=5, minval=1, maxval=10, stepval=0.5, desc="s1 Back Edges"),
    s6=new_slider(val=8, minval=1, maxval=10, stepval=0.5, desc="s1 Back Corners"),
    xo=new_slider(val=1, minval=0.9, maxval=1, stepval=0.05, desc="Xo Centering overall", update=True),
    p=new_slider(val=2.0, minval=1, maxval=3, stepval=0.2, desc="Exponent", update=True),
)

display(sliders)


interactive(children=(FloatSlider(value=6.0, continuous_update=False, description='s1 Front Surface', layout=L…